# 58. Causal Bayesian Optimization：介入を選び、実験で確かめる

このNotebookでは「工程条件を**観測する**」ことと「条件を実際に**操作する**」ことを区別し、既知DAGを持つ小さな合成工程で、**介入集合と値の提案 → 実験 → 観測登録 → GP再学習**を体験します。

**このNotebookで確認すること**
1. 既知DAGと構造方程式（SCM）から `do` 介入の結果を求める
2. `InterventionGPSurrogate` で集合別のGP事後を確認する
3. 既存BoTorch LogEIから介入集合と候補値を選ぶ
4. `CausalBOStudy.ask()` / `tell()` で本当に実験を追加する
5. 最良値と介入費用の違いを見る
6. GP-SCMの条件付きGP・do分布・MC-EI・Ask-and-Tellを可視化する

参考：[理論（do演算子・EIの式）](../../docs/theory/28_causal_bayesian_optimization.md) /
[実装ガイド（API内部の対応）](../../docs/guide/28_causal_bayesian_optimization.md) /
[比較Notebook](59_causal_bo_benchmark.ipynb)

**実行の流れ：** Known DAG → do/SCM → 介入データ → 集合別GP → fit/posterior → LogEI/Optimizer → Candidate → SCM実験 → tell → 再学習。実機の安全性や未知の因果構造の推定を扱う例ではありません。


## 1. 観測と介入の違いを体験する

今回は `X → M → Y ← Z` という既知のDAGを使います。XとZが工程で操作できる設定値、Mが中間の状態、Yが最大化したい目的出力です。

構造方程式は `M = 0.25 + 0.75 X`、`Y = -(M-0.8)^2 - 0.5(Z-0.65)^2` とします。式を1から書き直さず、Phase 6で用意した**既存の合成SCM**を再利用します。

`scm.evaluate()` は通常の生成機構を評価し、`scm.evaluate(HardIntervention({"X": ...}))` はXの構造方程式を置き換えて計算します。これは観測データを「同じ値の記録」でフィルタする操作とは違います。


In [ ]:
import torch
import matplotlib.pyplot as plt

from robotorchan.benchmarks import make_causal_synthetic_scenario
from robotorchan.causal import (
    CausalBOStudy,
    CausalObservation,
    HardIntervention,
    propose_causal_intervention,
)

torch.manual_seed(11)
scenario = make_causal_synthetic_scenario()
graph = scenario.graph
print("DAGのノード順序:", graph.topological_order)
print("Yの親:", graph.parents_of("Y"))
print("許された介入集合:", scenario.intervention_sets)


In [ ]:
natural = scenario.scm.evaluate()
do_x = HardIntervention({"X": 0.8})
under_do = scenario.scm.evaluate(do_x)
observed = CausalObservation(natural)                 # 観測だけ
intervened = CausalObservation(under_do, do_x)        # 実行した介入
print("通常の生成値:", natural)
print("do(X=0.8)の結果:", under_do)
print("is_interventional:", observed.is_interventional, intervened.is_interventional)
assert not observed.is_interventional
assert intervened.is_interventional
assert under_do["M"] != natural["M"]


## 2. 実験予算と介入集合を決める

介入集合は `{X}` と `{X,Z}` の2種類に絞ります（`{Z}` も既存APIでは選択可能です）。両者は異なる介入機構です。コストは `{X}:1`、`{X,Z}:3` とします。

`CausalBOStudy` は各集合で**成功した介入データが2件**集まるまではSobol初期候補を返します。初期化時にDAGを学習したり、因果構造を推定したりしません。最初の実験が評価可能なことは、今回のSCMを既知としているからです。


In [ ]:
study = CausalBOStudy(
    graph,
    "Y",
    intervention_sets=(("X",), ("X", "Z")),
    bounds=scenario.bounds,
    intervention_costs={
        frozenset({"X"}): 1.0,
        frozenset({"X", "Z"}): 3.0,
    },
    n_initial_trials=2,
    optimizer="sobol",   # 軽量な候補探索。獲得関数は既存のBoTorch LogEI
    raw_samples=16,
    seed=11,
)
for _ in range(4):
    trial = study.ask()
    measured = scenario.evaluate(trial.intervention)  # 実世界ならここで実験
    completed = study.tell(trial, measured.values)
    print("初期実験", completed.number, dict(completed.intervention.values),
          "Y =", round(completed.observation.values["Y"], 5))
print("成功した介入記録数:", len(study.observations))
assert len(study.observations) == 4


## 3. 介入GPを学習し、Posteriorを見る

`study.fit()` は **介入集合ごとに独立した** `InterventionGPSurrogate` を学習します。これはrobotorchanの既存 `SingleTaskGP.fit()` への委譲です。

`{X}` のGPにとって入力はXだけの1列 `[n,1]` です。 `{X,Z}` のGPは2列 `[n,2]` となります。**観測だけのレコードを介入GPに混ぜず、別の介入集合も混ぜません。**

Posteriorの`mean`は「その集合を介入したら期待される目的値」、`variance`はモデル予測の不確かさです。これらは因果方向の正しさを検証するものではありません。


In [ ]:
surrogates = study.fit()
for gp in surrogates:
    print("介入集合:", gp.intervention_nodes,
          "学習X:", tuple(gp.train_X.shape), "学習Y:", tuple(gp.train_Y.shape))
gp_x = surrogates[0]
with torch.no_grad():
    posterior = gp_x.posterior(torch.tensor([[0.55]], dtype=torch.double))
print("do(X=0.55) の予測平均:", posterior.mean.squeeze().item())
print("予測標準偏差:", posterior.variance.sqrt().squeeze().item())
assert len(surrogates) == 2


## 4. LogEIから「次にどの介入をするか」を提案する

`propose_causal_intervention` は、各GPの`train_Y`から**共通の現在の最良値**を作り、BoTorchの`LogExpectedImprovement`を既存Optimizerで最大化します。

各集合の介入コストは `log(EI) - log(cost)` として順位に反映されます。LogEIを費用で直接割るのではありません。提案スコアは**実測したYではない**ため、その候補の結果を知るには介入実験が必要です。


In [ ]:
proposal = propose_causal_intervention(
    surrogates,
    study.bounds,
    intervention_costs=study.intervention_costs,
    optimizer="sobol",
    raw_samples=16,
)
print("提案do:", dict(proposal.intervention.values))
print("LogEI:", proposal.log_expected_improvement)
print("介入コスト:", proposal.intervention_cost)
print("コスト調整後スコア:", proposal.log_utility)


## 5. Ask-and-Tellで2回の逐次介入実験を回す

冷スタートが終了しているため、この先の`ask()`はGPとLogEIを用いた候補提案に切り替わります。

- **ask**：介入集合と介入値を選択し、Trialを`RUNNING`にする
- **外部実験**：提案されたHard interventionを実施する（ここではSCMで代行）
- **tell**：介入した条件と実測Yを`COMPLETE`として登録する
- **次のask**：新しい記録を使って必要なGPを再学習する

`tell()`しないまま次の`ask()`を呼ぶことはできません。`state=TrialState.FAIL`で失敗も管理できますが、その場合はGPの学習データに入りません。


In [ ]:
for iteration in range(2):
    trial = study.ask()
    assert trial.proposal is not None  # 初期SobolからGPによる提案に移行
    outcome = scenario.evaluate(trial.intervention)  # 真のdo実験
    completed = study.tell(trial, {"Y": outcome.values["Y"]})
    print(
        f"Round {iteration + 1}:",
        "do =", dict(completed.intervention.values),
        "Y =", round(completed.observation.values["Y"], 5),
        "state =", completed.state.name,
    )
print("累積の介入観測件数:", len(study.observations))
assert len(study.observations) == 6


## 6. 最良値とコストの推移を別々に見る

この例は最大化なので、各実験のYを順に`max`で更新した値が最良値です。コストは各集合の固定費用を積算します。安い介入ばかり選んだときに、必ず最良値も高くなるわけではありません。

横軸が**実験回数**の場合と**累積コスト**の場合では、比較の公平性が違います。


In [ ]:
best = -float("inf")
spent = 0.0
best_values = []
costs = []
for trial in study.trials:
    assert trial.observation is not None
    key = frozenset(trial.intervention.values)
    best = max(best, trial.observation.values["Y"])
    spent += study.intervention_costs[key]
    best_values.append(best)
    costs.append(spent)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(range(1, len(best_values) + 1), best_values, marker="o")
axes[0].set(xlabel="Evaluation count", ylabel="Best observed Y", title="Best value")
axes[1].plot(costs, best_values, marker="o")
axes[1].set(xlabel="Cumulative intervention cost", ylabel="Best observed Y",
            title="Best value vs cost")
plt.tight_layout()
plt.show()
print("最終最良値:", best_values[-1], "累積コスト:", costs[-1])


## 7. GP-SCM：1つの因果モデルで複数の生成機構を学ぶ

ここからは**前半の集合別GPとは独立した例**を使います。既知DAGは `X → Z → Y`、
構造方程式は

\[
Z = 1.5X, \qquad Y = 0.3 + Z
\]

とします。今回の自然な `X` は環境により変動するrootです。以下の観測データはこの生成式に従って作った架空の**非介入記録**であり、前半の「rootが定数の合成ベンチマーク」とは別の実験系です。

因果モデルの意味を確認するため、`do(X=x)`では `X` を固定して `Z,Y` を伝播し、`do(Z=z)`では `Z` の方程式を置換します。この例では `Y` に直接影響するのは `Z` だけです。

In [ ]:
from robotorchan.causal import (
    CausalGraph,
    GPStructuralCausalModel,
    propose_gp_scm_intervention,
)

gp_graph = CausalGraph({"X": (), "Z": ("X",), "Y": ("Z",)})
x_values = torch.linspace(0.1, 0.9, 9, dtype=torch.double)
observational_records = [
    CausalObservation({"X": float(x), "Z": 1.5 * float(x), "Y": 0.3 + 1.5 * float(x)})
    for x in x_values
]
print("観測データ:", len(observational_records), "件")
print("Zの親:", gp_graph.parents_of("Z"), "Yの親:", gp_graph.parents_of("Y"))
assert all(not record.is_interventional for record in observational_records)

gp_scm = GPStructuralCausalModel(gp_graph, observational_records)
gp_scm.fit()
for name, mechanism in gp_scm.mechanisms.items():
    print("条件付きGP:", name, "親:", gp_graph.parents_of(name),
          "学習数:", mechanism.raw_train_X.shape[0])
assert set(gp_scm.mechanisms) == {"Z", "Y"}

## 8. 条件付きGPと `do` 予測を区別して可視化する

`mechanism_posterior("Y", X_Z)` は **`Z` が与えられた場合の `Y` の条件付きGP事後**です。一方、`sample_interventions(..., intervention_nodes=("X",))` は、
**`X` に介入した後に `Z` を経由して `Y` に到達する予測**です。

後者は必ずしも正規分布ではないため、BoTorch標準のPosteriorを装わず、
サンプルから平均・2.5%/97.5%分位点を計算します。これらはモデル仮定の下での
**予測分位点**であり、因果構造の妥当性まで保証する95%信頼区間ではありません。

In [ ]:
z_grid = torch.linspace(0.0, 1.5, 35, dtype=torch.double)
with torch.no_grad():
    conditional = gp_scm.mechanism_posterior("Y", z_grid[:, None])
    mean_conditional = conditional.mean.squeeze(-1)
    std_conditional = conditional.variance.clamp_min(0).sqrt().squeeze(-1)

x_grid = torch.linspace(0.05, 0.95, 35, dtype=torch.double)
with torch.no_grad():
    draws_x = gp_scm.sample_interventions(
        x_grid[:, None, None],
        intervention_nodes=("X",),
        outcome="Y",
        sample_shape=torch.Size([128]),
    )
draws_x = draws_x.squeeze(-1)   # [128, 35]
mean_do_x = draws_x.mean(dim=0)
lower_do_x = torch.quantile(draws_x, 0.025, dim=0)
upper_do_x = torch.quantile(draws_x, 0.975, dim=0)
assert draws_x.shape == (128, 35)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(z_grid.numpy(), mean_conditional.numpy(), label="GP mean: Y | Z")
axes[0].fill_between(
    z_grid.numpy(),
    (mean_conditional - 1.96 * std_conditional).numpy(),
    (mean_conditional + 1.96 * std_conditional).numpy(),
    alpha=0.2, label="Gaussian marginal ±1.96 std",
)
axes[0].plot(z_grid.numpy(), 0.3 + z_grid.numpy(), "--", label="True equation")
axes[0].set(xlabel="Observed Z", ylabel="Y", title="Conditional mechanism")
axes[0].legend()
axes[1].plot(x_grid.numpy(), mean_do_x.numpy(), label="GP-SCM MC mean")
axes[1].fill_between(x_grid.numpy(), lower_do_x.numpy(), upper_do_x.numpy(),
                     alpha=0.2, label="MC 2.5–97.5% quantiles")
axes[1].plot(x_grid.numpy(), 0.3 + 1.5 * x_grid.numpy(), "--", label="True do(X=x)")
axes[1].set(xlabel="Intervened X", ylabel="Y", title="Propagated do prediction")
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. 観測データは学習に使うが、改善基準は**実際の介入値**から取る

GP-SCMの機構学習には9件の観測データを利用できました。しかし、
それだけでは「何を操作して最良の結果を実際に得たか」は分かりません。
`CausalBOStudy(model_type="gp_scm")` は、最初に**1件の実介入**をSobolで提案し、
`tell()`で完了してからMC Expected Improvementへ移行します。

次の関数は教育用の架空の実験装置です。実験値は `HardIntervention` に従い、
**介入しないroot Xは自然に変動**します。実運用では実機の記録を渡してください。

In [ ]:
gp_study = CausalBOStudy(
    gp_graph,
    "Y",
    intervention_sets=(("X",), ("Z",)),
    bounds={"X": (0.0, 1.0), "Z": (0.0, 1.5)},
    model_type="gp_scm",
    observational_data=observational_records,
    optimizer="sobol",
    raw_samples=12,
    mc_samples=48,
    mc_seed=17,
    seed=17,
)
print("Studyの純観測記録数:", len(gp_study.observational_data))
print("Studyの実介入記録数:", len(gp_study.observations))

simulation_rng = torch.Generator().manual_seed(17)

def execute_demo_intervention(intervention):
    fixed = intervention.values
    x = fixed.get("X", float(torch.rand((), generator=simulation_rng)))
    z = fixed.get("Z", 1.5 * x)
    return {"X": x, "Z": z, "Y": 0.3 + z}

first_trial = gp_study.ask()
assert first_trial.proposal is None   # 最初の1件はSobol
first_measurement = execute_demo_intervention(first_trial.intervention)
gp_study.tell(first_trial, first_measurement)
print("実施した最初のdo:", dict(first_trial.intervention.values))
print("実測Y:", first_measurement["Y"])
assert len(gp_study.observations) == 1
assert len(gp_study.observational_data) == 9

## 10. MC Expected Improvementを図で理解する

今回の集合間比較では、**実測された介入の最良値**を `best_f` とします。
候補ごとのGP-SCM予測から

\[
\widehat{\mathrm{EI}}(x)
=\frac{1}{M}\sum_{m=1}^{M}\max(y^{(m)}(x)-best_f,0)
\]

を確認できます。次の図は**獲得関数の考え方を視覚化するためのサンプル集計**であり、
Notebook内に別の獲得関数クラスを実装していません。実際の候補最適化には
ライブラリの `propose_gp_scm_intervention` を使用します。
実装のMC-LogEIでは有限MC誤差やゼロ改善領域での勾配の平坦化が起こり得ます。

In [ ]:
best_f = max(record.values["Y"] for record in gp_study.observations)
fitted_study_scm = gp_study.fit()  # fit requires gradients; keep it outside no_grad
with torch.no_grad():
    mc_grid = fitted_study_scm.sample_interventions(
        x_grid[:, None, None],
        intervention_nodes=("X",), outcome="Y",
        sample_shape=torch.Size([128]),
    ).squeeze(-1)
mc_ei = (mc_grid - best_f).clamp_min(0).mean(dim=0)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(x_grid.numpy(), mc_ei.numpy(), label="MC EI for do(X=x)")
ax.set(xlabel="Intervened X", ylabel="Expected improvement",
       title="MC improvement relative to measured intervention")
ax.legend()
plt.tight_layout()
plt.show()

standalone = propose_gp_scm_intervention(
    gp_study.fit(),
    (("X",), ("Z",)),
    gp_study.bounds,
    outcome="Y",
    best_f=best_f,
    intervention_costs={frozenset({"X"}): 1.0, frozenset({"Z"}): 1.5},
    optimizer="sobol",
    raw_samples=12,
    mc_samples=48,
    mc_seed=17,
)
print("Studyなしの単発提案:", dict(standalone.intervention.values))
print("MC LogEI:", standalone.acquisition_value)
print("コスト調整スコア:", standalone.ranking_score)

## 11. GP-SCM方式のAsk-and-Tell：モデルの更新も確認する

`study.fit()` はGP-SCM方式で `GPStructuralCausalModel` を返します。
直前の `fit()` はキャッシュされ、**成功した `tell()` の後は新しい実測データで再学習**します。
失敗した試行は学習データに入りません。実験記録には操作した値だけでなく、
中間ノードの測定値 `Z` も含めます。

In [ ]:
before_refit = gp_study.fit()
selected_trial = gp_study.ask()
assert selected_trial.proposal is not None
print("最適化から選択したdo:", dict(selected_trial.intervention.values))
print("Trial state:", selected_trial.state)
measured = execute_demo_intervention(selected_trial.intervention)
completed = gp_study.tell(selected_trial, measured)
after_refit = gp_study.fit()
assert before_refit is not after_refit
assert len(gp_study.observations) == 2
print("登録した実測値:", dict(completed.observation.values))
print("更新後の実介入件数:", len(gp_study.observations))
print("純観測データは別管理:", len(gp_study.observational_data))

## 12. 2方式を使い分けるためのチェックリスト

- **集合別GP**：過去の実介入結果だけがある／中間ノードの測定が難しい場合に扱いやすい。ただし各集合ごとに初期実験が必要です。
- **GP-SCM**：既知DAGが信頼でき、親・中間・結果の値を測定できる場合、複数介入集合で機構の学習情報を共有できます。因果十分性と機構不変性の仮定が必要です。
- **共通**：`ask()`は提案のみ。実際の介入は外部の装置・実験担当者が実施します。未観測交絡の自動調整、DAGの発見、安全制約の保証、`q>1`の同時提案はありません。

より厳密な定義は[Theory 28章](../../docs/theory/28_causal_bayesian_optimization.md)、比較実験の限界は[Notebook 59](59_causal_bo_benchmark.ipynb)を参照してください。

## 13. 実務へ適用する前の注意

ここでは構造が既知の**決定的な合成SCM**なので、観測がどの介入から得られたかを完全に記録できます。実世界では、実験の実施ログ・入力に対する実際の操作・未操作変数の状態・測定雑音を明示してください。

現在のCBOは、既知DAG・集合別GPまたはGP-SCM・連続介入・単一目的・`q=1`・正の固定介入コストが前提です。GP-SCMは因果十分性・機構不変性および必要な親と中間ノードの測定を仮定します。**観測だけのデータから因果効果を同定したり、因果グラフを発見したりしません。** 保護装置や因果妥当性の検証を置き換えるものではありません。

比較を学びたい方は[59番Notebook](59_causal_bo_benchmark.ipynb)、数式は[Theory 28章](../../docs/theory/28_causal_bayesian_optimization.md)、API内部は[Guide 28章](../../docs/guide/28_causal_bayesian_optimization.md)を参照してください。
